In [ ]:
# SAME-STATION DAILY HORIZONTAL ACF TENSORS
#
# IMPORTANT:
# - SAME station autocorrelations
# - Uses adjacent-station azimuth/backazimuth
# - SAVES ONE .NPZ PER DAY
# - NO FINAL STACKING
# - NO SavGol
# - NO common-mode removal
# =========================================================

import os
import glob
import time
import warnings
import numpy as np
from tqdm import tqdm
import scipy.signal as signal
from scipy.signal import hilbert
from scipy import fftpack

from obspy import read
from obspy import read_inventory
from obspy.geodetics import gps2dist_azimuth

warnings.filterwarnings("ignore")

# =========================================================
# PATHS
# =========================================================

ROOT = r"D:\Alan CUSAT- dissertation\TRANSCORBE\9H"

OUTPUT_ROOT = (
    r"D:\Alan CUSAT- dissertation\TRANSCORBE"
    r"\HORIZONTAL_ACF_TENSOR_DAILY"
)

STATION_XML = (
    r"D:\Alan CUSAT- dissertation\TRANSCORBE"
    r"\stations.xml"
)

os.makedirs(OUTPUT_ROOT, exist_ok=True)

# =========================================================
# PARAMETERS
# =========================================================

TARGET_DT = 0.2

FREQMIN = 0.1
FREQMAX = 2.0

MAX_LAG_SEC = 100

MIN_OVERLAP_HOURS = 12

MIN_SAMPLES = 1000

# =========================================================
# PROFILE ORDER
# =========================================================

PROFILE_ORDER = [
    "TR02","TR03","TR04","TR05",
    "TR08","TR10","TR11","TR12",
    "TR13","TR14","TR15","TR16",
    "TR17","TR18","TR19","TR20",
    "TR21","TR22","TR23","TR24",
    "TR25","TR26","TR27","TR28",
    "TR29","TR30","TR31","TR32",
    "TR34","TR35","TR36","TR37",
    "TR38","TR39","TR40","TR41",
    "TR42","TR43",
]

# =========================================================
# EXCLUDE TEMP STATIONS
# =========================================================

EXCLUDE_STATIONS = {
    "TR27A",
    "TR28A",
    "TR29A",
    "TR32N"
}

# =========================================================
# LOAD STATION METADATA
# =========================================================

inv = read_inventory(STATION_XML)

coords = {}

for net in inv:
    for sta in net:
        coords[sta.code] = {
            "lat": sta.latitude,
            "lon": sta.longitude
        }

# =========================================================
# FIND AVAILABLE STATIONS
# =========================================================

available_stations = set()

for year_dir in glob.glob(os.path.join(ROOT, "20*")):

    stations_found = glob.glob(
        os.path.join(year_dir, "TR*")
    )

    for s in stations_found:
        available_stations.add(
            os.path.basename(s)
        )

stations = [
    s for s in PROFILE_ORDER
    if (
        s in available_stations
        and s not in EXCLUDE_STATIONS
    )
]

print("\n========================================")
print("STATIONS")
print("========================================")

for s in stations:
    print(s)

# =========================================================
# ADJACENT-STATION AZIMUTH TABLE
# =========================================================

adjacent_geometry = {}

for i in range(len(stations)-1):

    sta1 = stations[i]
    sta2 = stations[i+1]

    lat1 = coords[sta1]["lat"]
    lon1 = coords[sta1]["lon"]

    lat2 = coords[sta2]["lat"]
    lon2 = coords[sta2]["lon"]

    _, az12, baz12 = gps2dist_azimuth(
        lat1,
        lon1,
        lat2,
        lon2
    )

    adjacent_geometry[sta1] = {
        "az": az12,
        "baz": baz12
    }

adjacent_geometry[stations[-1]] = (
    adjacent_geometry[stations[-2]]
)

# =========================================================
# BASIC FUNCTIONS
# =========================================================

def safe_read(filepath):

    try:
        st = read(filepath)

        if len(st) == 0:
            return None

        return st[0]

    except:
        return None


def preprocess(tr):

    tr = tr.copy()

    if abs(tr.stats.delta - TARGET_DT) > 1e-5:
        tr.resample(1 / TARGET_DT)

    tr.detrend("linear")
    tr.detrend("demean")

    tr.filter(
        "bandpass",
        freqmin=FREQMIN,
        freqmax=FREQMAX,
        zerophase=True
    )

    tr.taper(0.05)

    return tr


def comb_filter(trace, fs):

    p1 = [0.5, 2, 6]
    p2 = [2, 6, 20]

    nyq = fs / 2

    weights = []

    for T1, T2 in zip(p1, p2):

        fmax = 1.0 / T1
        fmin = 1.0 / T2

        if fmax >= nyq:
            continue

        b, a = signal.butter(
            2,
            [fmin, fmax],
            btype="band",
            fs=fs
        )

        trf = signal.filtfilt(
            b,
            a,
            trace
        )

        env = np.abs(hilbert(trf))

        weights.append(
            1.0 / (env + 1e-10)
        )

    if len(weights) == 0:

        return (
            trace
            / (np.max(np.abs(trace)) + 1e-10)
        )

    W = np.ones_like(trace)

    for w in weights:
        W *= w

    water_level = 0.10 * np.mean(W)

    W[W < water_level] = water_level

    trace_norm = trace * W

    return (
        trace_norm
        / (np.max(np.abs(trace_norm)) + 1e-10)
    )


def crosscorrelation(trace01, trace02, maxlag):

    lentrace = len(trace01)

    goodnumber = int(maxlag + lentrace)

    tr2 = np.zeros(goodnumber)

    tr2[0:lentrace] = trace02

    tr2[0:lentrace] /= np.sqrt(
        np.sum(tr2[0:lentrace] ** 2)
    )

    tr2 = fftpack.fft(
        tr2,
        overwrite_x=True
    )

    tr2.imag *= -1

    tr1 = np.zeros(goodnumber)

    tr1[maxlag:maxlag+lentrace] = trace01

    tr1[maxlag:maxlag+lentrace] /= np.sqrt(
        np.sum(
            tr1[maxlag:maxlag+lentrace] ** 2
        )
    )

    tr2 *= fftpack.fft(
        tr1,
        overwrite_x=True
    )

    return fftpack.ifft(
        tr2,
        overwrite_x=True
    )[0:2*maxlag+1].real.astype(np.float32)


def symmetrize_ccf(ccf):

    return 0.5 * (ccf + ccf[::-1])


def normalize(x):

    return (
        x / (np.max(np.abs(x)) + 1e-10)
    )

# =========================================================
# ROTATION
# =========================================================

def rotate_tensor(EE, EN, NN, NE, az, baz):

    theta = np.deg2rad(az)
    psi   = np.deg2rad(baz)

    ct = np.cos(theta)
    st = np.sin(theta)

    cp = np.cos(psi)
    sp = np.sin(psi)

    TT = (
        -ct * cp * EE
        + ct * sp * EN
        - st * sp * NN
        + st * cp * NE
    )

    RR = (
        -st * sp * EE
        -st * cp * EN
        -ct * cp * NN
        -ct * sp * NE
    )

    TR = (
        -ct * sp * EE
        -ct * cp * EN
        +st * cp * NN
        +st * sp * NE
    )

    RT = (
        -st * cp * EE
        +st * sp * EN
        +ct * sp * NN
        -ct * cp * NE
    )

    return RR, RT, TR, TT

# =========================================================
# BUILD FILE DATABASE
# =========================================================

print("\n========================================")
print("INDEXING DATA")
print("========================================")

file_db = {}

for station in stations:
    file_db[station] = {}

for year_dir in glob.glob(os.path.join(ROOT, "20*")):

    for station in stations:

        for comp in ["HHE", "HHN"]:

            comp_dir = os.path.join(
                year_dir,
                station,
                comp
            )

            if not os.path.exists(comp_dir):
                continue

            files = glob.glob(
                os.path.join(comp_dir, "*.mseed")
            )

            for f in files:

                try:

                    fname = os.path.basename(f)

                    date_str = fname.split("_")[1]

                    if date_str not in file_db[station]:
                        file_db[station][date_str] = {}

                    file_db[station][date_str][comp] = f

                except:
                    continue

# =========================================================
# COUNT TOTAL DAYS
# =========================================================

TOTAL_JOBS = 0

for sta in stations:
    TOTAL_JOBS += len(file_db[sta])

print("\n========================================")
print(f"TOTAL DAYS = {TOTAL_JOBS}")
print("========================================")

# =========================================================
# MAIN PROCESSING
# =========================================================

start_total = time.time()

pbar = tqdm(
    total=TOTAL_JOBS,
    ncols=120,
    ascii=False
)

processed_jobs = 0

for sta in stations:

    outdir = os.path.join(
        OUTPUT_ROOT,
        sta
    )

    os.makedirs(outdir, exist_ok=True)

    az  = adjacent_geometry[sta]["az"]
    baz = adjacent_geometry[sta]["baz"]

    station_days = sorted(
        file_db[sta].keys()
    )

    for date_str in station_days:

        processed_jobs += 1

        pbar.update(1)

        try:

            outfile = os.path.join(
                outdir,
                f"{date_str}.npz"
            )

            # =============================================
            # SKIP EXISTING FILES
            # =============================================

            if os.path.exists(outfile):
                continue

            files = file_db[sta][date_str]

            if (
                "HHE" not in files
                or "HHN" not in files
            ):
                continue

            E = safe_read(files["HHE"])
            N = safe_read(files["HHN"])

            if E is None or N is None:
                continue

            E = preprocess(E)
            N = preprocess(N)

            start = max(
                E.stats.starttime,
                N.stats.starttime
            )

            end = min(
                E.stats.endtime,
                N.stats.endtime
            )

            overlap_hours = (
                end - start
            ) / 3600.0

            if overlap_hours < MIN_OVERLAP_HOURS:
                continue

            E.trim(start, end)
            N.trim(start, end)

            if len(E.data) < MIN_SAMPLES:
                continue

            fs = E.stats.sampling_rate
            dt = E.stats.delta

            maxlag = int(MAX_LAG_SEC * fs)

            Edata = comb_filter(
                E.data.astype(np.float32),
                fs
            )

            Ndata = comb_filter(
                N.data.astype(np.float32),
                fs
            )

            # =================================================
            # SAME-STATION AUTOCORRELATIONS
            # =================================================

            EE = crosscorrelation(
                Edata,
                Edata,
                maxlag
            )

            EN = crosscorrelation(
                Edata,
                Ndata,
                maxlag
            )

            NN = crosscorrelation(
                Ndata,
                Ndata,
                maxlag
            )

            NE = crosscorrelation(
                Ndata,
                Edata,
                maxlag
            )

            EE = symmetrize_ccf(EE)
            EN = symmetrize_ccf(EN)
            NN = symmetrize_ccf(NN)
            NE = symmetrize_ccf(NE)

            RR, RT, TR, TT = rotate_tensor(
                EE,
                EN,
                NN,
                NE,
                az,
                baz
            )

            RR = normalize(RR)
            RT = normalize(RT)
            TR = normalize(TR)
            TT = normalize(TT)

            lags = np.arange(
                -maxlag,
                maxlag + 1
            ) * dt

            # =================================================
            # SAVE DAILY FILE
            # =================================================

            np.savez_compressed(

                outfile,

                RR=RR.astype(np.float32),
                RT=RT.astype(np.float32),
                TR=TR.astype(np.float32),
                TT=TT.astype(np.float32),

                lags=lags.astype(np.float32),

                dt=np.float32(dt),

                station=sta,

                azimuth=np.float32(az),
                backazimuth=np.float32(baz),

                overlap_hours=np.float32(
                    overlap_hours
                ),

                date=date_str
            )

        except:
            continue

        elapsed = (
            time.time() - start_total
        )

        avg_time = (
            elapsed / processed_jobs
        )

        remaining_jobs = (
            TOTAL_JOBS - processed_jobs
        )

        remaining_sec = (
            remaining_jobs * avg_time
        )

        pbar.set_postfix({
            "station": sta,
            "ETA(hr)": f"{remaining_sec/3600:.1f}",
        }, refresh=False)

pbar.close()

# =========================================================
# SUMMARY
# =========================================================

total_time = (
    time.time() - start_total
)

print("\n========================================")
print("DONE")
print(f"TOTAL TIME = {total_time/3600:.2f} hr")
print("========================================")


STATIONS
TR02
TR03
TR04
TR05
TR08
TR10
TR11
TR12
TR13
TR14
TR15
TR16
TR17
TR18
TR19
TR20
TR21
TR22
TR23
TR24
TR25
TR26
TR27
TR28
TR29
TR30
TR31
TR32
TR34
TR35
TR36
TR37
TR38
TR40
TR41
TR42
TR43

INDEXING DATA

TOTAL DAYS = 23430


  0%|                                                   | 38/23430 [00:41<7:10:12,  1.10s/it, station=TR02, ETA(hr)=7.2]